In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import json
import time
import re
from google.colab import drive


# --- 2. CẤU HÌNH ---
JSON_FOLDER = "/content/drive/MyDrive/wiki_data_json_final"
ROOT_SAVE_DIR = "/content/drive/MyDrive/Wiki_2_Images"
LOG_FILE = "/content/drive/MyDrive/Wiki_2_Images/processed_log.txt"
MIN_SIZE = 50

# Danh sách các quốc gia đã cào xong (Dựa trên ảnh bạn gửi)
SKIP_COUNTRIES = {
    "Viet_Nam"
}

HEADERS = {
    'User-Agent': 'WikiExplorer/5.0 (contact: yourname@email.com) DataGatheringBot'
}

def clean_name(text):
    if not text: return "Unknown"
    return re.sub(r'[\\/*?:"<>|]', '', text).strip().replace(' ', '_')

def get_full_res_url(url):
    if not url: return None
    full_url = 'https:' + url if url.startswith('//') else url
    if '/thumb/' in full_url:
        parts = full_url.replace('/thumb/', '/').split('/')
        return '/'.join(parts[:-1])
    return full_url

def is_valid_image(response):
    content_type = response.headers.get('Content-Type', '')
    if 'image' not in content_type:
        return False
    if b"<!DOCTYPE html" in response.content[:100]:
        return False
    return True

def start_mining():
    if not os.path.exists(ROOT_SAVE_DIR): os.makedirs(ROOT_SAVE_DIR)

    # Đọc log để bỏ qua các URL lẻ đã hoàn thành
    processed_urls = set()
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8') as f:
            processed_urls = set(line.strip() for line in f)

    json_files = sorted([f for f in os.listdir(JSON_FOLDER) if f.endswith('.json')])

    for j_file in json_files:
        print(f"\n Đang xử lý file: {j_file}")
        with open(os.path.join(JSON_FOLDER, j_file), 'r', encoding='utf-8') as f:
            try: data = json.load(f)
            except: continue

        for item in data:
            country_raw = item.get('country', 'Unknown')
            country = clean_name(country_raw)

            # --- KIỂM TRA QUỐC GIA ĐÃ XONG ---
            if country in SKIP_COUNTRIES:
                continue

            title = clean_name(item.get('title', 'UnknownPlace'))
            wiki_url = item.get('url')

            if not wiki_url or wiki_url in processed_urls:
                continue

            country_dir = os.path.join(ROOT_SAVE_DIR, country)
            os.makedirs(country_dir, exist_ok=True)

            # Check file thực tế đề phòng log chưa ghi
            if os.path.exists(os.path.join(country_dir, f"{country}_{title}_0.jpg")):
                continue

            print(f" Vét ảnh: {title} ({country})")

            try:
                time.sleep(1.5)
                res = requests.get(wiki_url, headers=HEADERS, timeout=15)
                if res.status_code == 429:
                    print(" Bị chặn 429! Nghỉ 3 phút...")
                    time.sleep(180)
                    continue
                if res.status_code != 200: continue

                soup = BeautifulSoup(res.text, 'html.parser')
                content = soup.find(id="mw-content-text")
                if not content: continue

                imgs = content.find_all('img')
                downloaded_urls = set()
                count = 0
                # cais nay la lay full anh
                # for img in imgs:
                #     try:
                #         w, h = int(img.get('width', 0)), int(img.get('height', 0))
                #         if w < MIN_SIZE and h < MIN_SIZE: continue

                #         img_url = get_full_res_url(img.get('src'))
                #         if not img_url or img_url in downloaded_urls: continue

                #         time.sleep(0.8)
                #         img_res = requests.get(img_url, headers=HEADERS, timeout=15)

                #         if img_res.status_code == 200 and is_valid_image(img_res):
                #             ext = img_url.split('.')[-1].lower().split('?')[0]
                #             if len(ext) > 4: ext = 'jpg'

                #             filename = f"{country}_{title}_{count}.{ext}"
                #             with open(os.path.join(country_dir, filename), 'wb') as f:
                #                 f.write(img_res.content)

                #             downloaded_urls.add(img_url)
                #             count += 1
                #         elif img_res.status_code == 429:
                #             print(" Bị chặn khi tải ảnh! Nghỉ 2 phút...")
                #             time.sleep(120)
                #             break
                #     except: continue

                # laays 2 anh thui
                for img in imgs:
                    # Nếu đã tải đủ 2 ảnh thì dừng quét các ảnh còn lại của địa danh này
                    if count >= 2:
                        break

                    try:
                        w, h = int(img.get('width', 0)), int(img.get('height', 0))
                        if w < MIN_SIZE and h < MIN_SIZE: continue

                        img_url = get_full_res_url(img.get('src'))
                        if not img_url or img_url in downloaded_urls: continue

                        time.sleep(0.8)
                        img_res = requests.get(img_url, headers=HEADERS, timeout=15)

                        if img_res.status_code == 200 and is_valid_image(img_res):
                            ext = img_url.split('.')[-1].lower().split('?')[0]
                            if len(ext) > 4: ext = 'jpg'

                            filename = f"{country}_{title}_{count}.{ext}"
                            with open(os.path.join(country_dir, filename), 'wb') as f:
                                f.write(img_res.content)

                            downloaded_urls.add(img_url)
                            count += 1 # Tăng biến đếm sau mỗi lần tải thành công

                        elif img_res.status_code == 429:
                            print(" Bị chặn khi tải ảnh! Nghỉ 2 phút...")
                            time.sleep(120)
                            break
                    except:
                        continue

                # Ghi log hoàn tất địa danh
                with open(LOG_FILE, 'a', encoding='utf-8') as f:
                    f.write(wiki_url + '\n')
                processed_urls.add(wiki_url)

                if count > 0: print(f"    Đã lấy {count} ảnh.")

            except Exception as e:
                print(f"    Lỗi: {e}")
                continue

if __name__ == "__main__":
    start_mining()